В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [13]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [14]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [15]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [16]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [17]:
import gensim.downloader as api
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [18]:
model = api.load("glove-twitter-100")

In [19]:
print(f"Размер словаря: {len(model.key_to_index)}")
print(f"Размерность векторов: {model.vector_size}")

Размер словаря: 1193514
Размерность векторов: 100


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [21]:
n = 15
for i, word in enumerate(model.key_to_index.keys()):
    if i >= n:
        break
    vector = model[word]
    print(f"{i+1}. Слово: {word}")

word = str(input())
vector = model[word]
print(f"Вектор слова '{word}': {vector[:5]}...")

similar_words = model.most_similar(word, topn=5)
print(f"Слова, похожие на '{word}':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

1. Слово: <user>
2. Слово: .
3. Слово: :
4. Слово: rt
5. Слово: ,
6. Слово: <repeat>
7. Слово: <hashtag>
8. Слово: <number>
9. Слово: <url>
10. Слово: !
11. Слово: i
12. Слово: a
13. Слово: "
14. Слово: the
15. Слово: ?
break
Вектор слова 'break': [-0.38668   0.096059  0.55491  -0.72268   0.15098 ]...
Слова, похожие на 'break':
  time: 0.8177
  then: 0.7880
  end: 0.7873
  before: 0.7848
  take: 0.7791


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [22]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [23]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1,
    epochs=100,
    seed=42
)


In [24]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [25]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  горшок: 0.2245
  салат: 0.2234
  резать: 0.2137
  картофель: 0.2102
  соус: 0.2086


In [26]:
# Найдите слова, похожие на "духовка"
### ваш код здесь ###
try:
    similar = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")
# Найдите слова, похожие на "овощи"
### ваш код здесь ###
try:
    similar = model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'духовка':
  барбекю: 0.3331
  мариновать: 0.3065
  специи: 0.2847
  молоко: 0.2550
  жарить: 0.2186
Слова, похожие на 'овощи':
  вино: 0.3390
  взбивать: 0.2623
  сковорода: 0.2549
  бекон: 0.2505
  рыба: 0.2454


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [27]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [30]:
def find(word):
  try:
    similar = ft_model.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
  except KeyError:
    print(f"Слово '{word}' не найдено в словаре")
  return ''

words = ['варить', 'духовка', 'овощи']
for word in words:
  print(find(word))

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094



7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [32]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('лемон')
compare_models('овоши')
compare_models('вада')


Сравнение для слова: 'лемон'
  Word2Vec: слово не найдено
  FastText: ['сковорода', 'масло']

Сравнение для слова: 'овоши'
  Word2Vec: слово не найдено
  FastText: ['чай', 'яичница']

Сравнение для слова: 'вада'
  Word2Vec: слово не найдено
  FastText: ['уголь', 'вода']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [33]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [34]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [35]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [38]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")
    return similarity

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


np.float32(-0.08219737)

8. Сравните схожесть doc_2 и doc_4

In [39]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


np.float32(-0.0362442)

9. Найдите самый похожий документ на doc_1

In [43]:
mark_list = {}
for n in [0, 2, 3, 4]:
  res = compare_documents(1, n)
  mark_list[n] = res

max_mark = max(mark_list.values())
for n, mark in mark_list.items():
  if max_mark == mark:
    print(f'Самый похожий документ на doc_1: doc_{n}')


Схожесть doc_1 и doc_0: 0.2735
  doc_1: deep learning uses neural networks
  doc_0: machine learning is interesting
Схожесть doc_1 и doc_2: -0.0573
  doc_1: deep learning uses neural networks
  doc_2: python programming for data science
Схожесть doc_1 и doc_3: 0.2031
  doc_1: deep learning uses neural networks
  doc_3: artificial intelligence is amazing
Схожесть doc_1 и doc_4: -0.2546
  doc_1: deep learning uses neural networks
  doc_4: computer vision processes images
Самый похожий документ на doc_1: doc_0


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [48]:
model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

test = ['морозилка', 'разрыхлитель', 'томить']

print('Результат для модели с размерностью 10:')
for word in test:
    similar = model_10.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")

print('Результат для модели с размерностью 50:')
for word in test:
    similar = model_50.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")

print('Результат для модели с размерностью 100:')
for word in test:
    similar = model_100.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")

Результат для модели с размерностью 10:
Слова, похожие на 'морозилка':
  парить: 0.5663
  картофель: 0.5140
  вино: 0.4752
  жарить: 0.4184
  кипятить: 0.4177
Слова, похожие на 'разрыхлитель':
  пирог: 0.8478
  резать: 0.4858
  яичница: 0.4755
  рыба: 0.3983
  холодильник: 0.3940
Слова, похожие на 'томить':
  яблоки: 0.6350
  лимон: 0.5313
  готовить: 0.5305
  яичница: 0.5289
  ингредиенты: 0.4790
Результат для модели с размерностью 50:
Слова, похожие на 'морозилка':
  питание: 0.4653
  ингредиенты: 0.3166
  завтрак: 0.2776
  начинка: 0.2620
  месить: 0.2428
Слова, похожие на 'разрыхлитель':
  масло: 0.3485
  горшок: 0.3122
  дрожжи: 0.2261
  чашка: 0.2157
  завтрак: 0.2119
Слова, похожие на 'томить':
  варить: 0.3726
  жарить: 0.3091
  парить: 0.3000
  кипятить: 0.2919
  начинка: 0.2290
Результат для модели с размерностью 100:
Слова, похожие на 'морозилка':
  кипятить: 0.2299
  тушить: 0.2149
  курица: 0.1942
  рыба: 0.1882
  лимон: 0.1792
Слова, похожие на 'разрыхлитель':
  кофе: 0.2